In [1]:
## Burada 3 tool kullanacağız 
##internet için tavily
## bir RAG
## ve kendi kodumuz

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [5]:
search_tool = TavilySearchResults()

In [6]:
search_tool.invoke("2025 de oynanan Fenerbahçe Çayhur rizespor maçının sonucu nedir")

[{'url': 'https://www.espn.com/soccer/match/_/gameId/714087/caykur-rizespor-fenerbahce',
  'content': "Team | GP | W | D | L | GD | P\nGalatasaray | 30 | 24 | 5 | 1 | +44 | 77\nFenerbahce | 30 | 22 | 6 | 2 | +47 | 72\nSamsunspor | 31 | 15 | 6 | 10 | +8 | 51\nEyupspor | 31 | 14 | 8 | 9 | +15 | 50\nBesiktas | 30 | 13 | 10 | 7 | +12 | 49\nIstanbul Basaksehir | 30 | 14 | 6 | 10 | +10 | 48\nTrabzonspor | 30 | 11 | 9 | 10 | +12 | 42\nGaziantep FK | 30 | 12 | 6 | 12 | 0 | 42\nKasimpasa | 31 | 10 | 12 | 9 | -2 | 42\nGoztepe | 30 | 10 | 10 | 10 | +9 | 40\nKonyaspor | 31 | 11 | 7 | 13 | -5 | 40 [...] Fenerbahce\n\nCaykur Rizespor\n\nMatch Formations\n\nNo.Name\n1Irfan Can Egribayat\n33Diego Carlos23Cenk Tosun\n22Munir Mercan94Anderson Talisca\n37Milan Skriniar\n13Fred\n34Sofyan Amrabat10Dusan Tadic\n53Sebastian Szymanski\n18Filip Kostic17Irfan Can Kahveci\n70Oguz Aydin\n9Edin Dzeko\n19Youssef En-Nesyri\nSubstitutes\n8Mert Yandas\n54Osman Cetin\n21Bright Osayi-Samuel\n6Alexander Djiku\n5Ismail Yü

In [7]:
import bs4
#from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
loader = TextLoader("LOR.txt")

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

In [9]:
retriever.get_relevant_documents("Who made the one ring")[0]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_9900\1351436844.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  retriever.get_relevant_documents("Who made the one ring")[0]


Document(metadata={'source': 'LOR.txt'}, page_content="The title refers to the story's main antagonist,[b] the Dark Lord Sauron, who in an earlier age created the One Ring, allowing him to rule the other Rings of Power given to men, dwarves, and elves, in his campaign to conquer all of Middle-earth. From homely beginnings in the Shire, a hobbit land reminiscent of the English countryside, the story ranges across Middle-earth, following the quest to destroy the One Ring, seen mainly through the eyes of the hobbits Frodo, Sam, Merry, and Pippin. Aiding the hobbits are the wizard Gandalf, the men Aragorn and Boromir, the elf Legolas, and the dwarf Gimli, who unite as the Company of the Ring in order to rally the Free Peoples of Middle-earth against Sauron's armies and give Frodo a chance to destroy the One Ring in the fires of Mount Doom.")

In [10]:
from langchain.tools.retriever import create_retriever_tool

In [11]:
retriever_tool = create_retriever_tool(
    retriever,
    "Lord_Of_The_rings_search",
    "Search for information about Lord Of the rings. For any questions about Lord Of the Rings, you must use this tool!",
)

In [12]:
from langchain.agents import tool

In [13]:
@tool
def get_word_length_tool(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

In [14]:
get_word_length_tool.invoke("Ulas")

4

In [15]:
tools = [search_tool, retriever_tool, get_word_length_tool]

In [16]:
#Langchain birçok işlem için başarılı hazır prompt lar oluşturup bunları kullandırıyor
#liste için
#https://smith.langchain.com/hub/hwchase17
#bizde hazır birr prompt kullanacağız

In [17]:
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

C:\GenerativeAIEgitim\Env1\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant')),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [18]:
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)

In [19]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [20]:
agent_executor.invoke({"input": "hi! who are you?"})



> Entering new AgentExecutor chain...
Hello! I'm an AI assistant here to help you with information, answer questions, and assist with various tasks. How can I help you today?

> Finished chain.


{'input': 'hi! who are you?',
 'output': "Hello! I'm an AI assistant here to help you with information, answer questions, and assist with various tasks. How can I help you today?"}

In [21]:
agent_executor.invoke({"input": "Who made the one ring?"})



> Entering new AgentExecutor chain...

Invoking: `Lord_Of_The_rings_search` with `{'query': 'who made the one ring'}`


The title refers to the story's main antagonist,[b] the Dark Lord Sauron, who in an earlier age created the One Ring, allowing him to rule the other Rings of Power given to men, dwarves, and elves, in his campaign to conquer all of Middle-earth. From homely beginnings in the Shire, a hobbit land reminiscent of the English countryside, the story ranges across Middle-earth, following the quest to destroy the One Ring, seen mainly through the eyes of the hobbits Frodo, Sam, Merry, and Pippin. Aiding the hobbits are the wizard Gandalf, the men Aragorn and Boromir, the elf Legolas, and the dwarf Gimli, who unite as the Company of the Ring in order to rally the Free Peoples of Middle-earth against Sauron's armies and give Frodo a chance to destroy the One Ring in the fires of Mount Doom.

Gandalf proves that Frodo's Ring is the One Ring by throwing it into Frodo's firepla

{'input': 'Who made the one ring?',
 'output': 'The One Ring was created by the Dark Lord Sauron in an earlier age. He forged it to gain control over the other Rings of Power that were given to Men, Dwarves, and Elves, as part of his plan to conquer all of Middle-earth.'}

In [22]:
agent_executor.invoke({"input": "2025 de oynanan Fenerbahçe Çayhur rizespor maçının sonucu nedir?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '2025 Fenerbahçe Çaykur Rizespor maçı sonucu'}`


[{'url': 'https://beinsports.com.tr/haber/fenerbahce-caykur-rizespor-9', 'content': "08:50Trendyol Süper Lig'de 33. haftanın sakat ve cezalı raporu burada\n\nGENİŞ ÖZET | Fenerbahçe 3-2 Çaykur Rizespor\n\nYayınlanma Tarihi 2 Şubat 2025 Paz 20:59\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 19:06\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 18:31\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 18:21\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 18:18\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 18:15\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 15:56\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 15:08\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 14:53 [...] Güncelleme Tarihi 2 Şubat 2025 Paz 09:41\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 07:16\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 07:13\n\nGüncelleme Tarihi 2 Şubat 2025 Paz 07:13\n\nVideo yükleniyor...\n\nÖZET | Çaykur Rizespor 1-0 ikas Eyüpspor\n\nGENİŞ Ö

{'input': '2025 de oynanan Fenerbahçe Çayhur rizespor maçının sonucu nedir?',
 'output': "2025'te oynanan Fenerbahçe - Çaykur Rizespor maçı, Fenerbahçe'nin 3-2 galibiyetiyle sonuçlandı. Çaykur Rizespor maçta 2-0 öne geçti, ancak Fenerbahçe, Edin Dzeko'nun 45+8 ve 79. dakikada attığı gollerle ve En-Nesyri'nin 89. dakikada kaydettiği golle maçı çevirdi. Ayrıca, Çaykur Rizespor maçta iki kırmızı kart gördü ve 9 kişi kaldı. \n\nDaha fazla bilgi için [buraya](https://beinsports.com.tr/haber/fenerbahce-caykur-rizespor-9) göz atabilirsiniz."}

In [23]:
agent_executor.invoke({"input": "Ulaş isminde kaç harf var?"})



> Entering new AgentExecutor chain...

Invoking: `get_word_length_tool` with `{'word': 'Ulaş'}`


4"Ulaş" isminde 4 harf var.

> Finished chain.


{'input': 'Ulaş isminde kaç harf var?', 'output': '"Ulaş" isminde 4 harf var.'}

In [24]:
agent_executor.invoke({"input": "Fotosentez nedir?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'Fotosentez nedir?'}`


[{'url': 'https://tr.wikipedia.org/wiki/Fotosentez', 'content': "Fotosentez, bitkiler ve diğer canlılar tarafından, ışık enerjisini organizmaların yaşamsal eylemlerine enerji sağlamak için daha sonra serbest bırakılabilecek kimyasal enerjiye dönüştürmek için kullanılan bir işlemdir. Bu kimyasal enerji, karbondioksit ve sudan sentezlenen şekerler gibi karbonhidrat moleküllerinde depolanır. [...] Özellikleri\n\nFotosentez sözcüğü, Yunanca phōs (ışık) ve sentez (bir araya getirmek) sözcüklerinin bir araya getirilmesi ile oluşturulmuştur.\n\nÇoğu durumda oksijen yan ürün olarak salınır. Çoğu bitki, çoğu alg ve siyanobakteri fotosentez yapar; bu tür canlılara fotoototroflar denir. Fotosentez, Dünya atmosferinin oksijen içeriğinin üretilmesinden ve korunmasından büyük ölçüde sorumludur ve tüm organik bileşikler ve Dünya'daki yaşam için gerekli enerjinin çoğunu sağlar. [...]

{'input': 'Fotosentez nedir?',
 'output': 'Fotosentez, bitkiler, algler ve bazı bakteriler tarafından güneş ışığındaki enerjiyi kimyasal enerjiye dönüştüren biyokimyasal bir süreçtir. Bu süreçte, bitkiler karbondioksit ve su kullanarak glikoz (şeker) üretirler. Glikoz, bitkilerin enerji ihtiyacını karşılamak için depoladığı bir enerji kaynağıdır. Fotosentez sırasında, oksijen de yan ürün olarak serbest bırakılır.\n\nFotosentez kelimesi, Yunanca "phōs" (ışık) ve "sentez" (bir araya getirmek) kelimelerinin bir araya gelmesiyle oluşmuştur. Bu süreç, Dünya\'nın atmosferindeki oksijenin korunmasında ve tüm organik bileşiklerin üretiminde önemli bir rol oynamaktadır.\n\nFotosentezin genel biyokimyasal formülü şu şekildedir:\n\\[ 6 \\text{CO}_2 + 6 \\text{H}_2\\text{O} + \\text{ışık enerjisi} \\rightarrow \\text{C}_6\\text{H}_{12}\\text{O}_6 + 6 \\text{O}_2 \\]\n\nBu süreç, bitkilerin kloroplast adı verilen özel hücre organellerinde gerçekleşir. Kloroplastlar, ışık enerjisini emen klorofil pi

In [31]:
search_tool2 = TavilySearchResults(max_results=2, description="Sadece tarih, haber ve güncel olaylar için kullan")

In [32]:
tools2 = [search_tool2, retriever_tool, get_word_length_tool]

In [33]:
agent2 = create_tool_calling_agent(llm, tools2, prompt)

In [34]:
agent_executor2 = AgentExecutor(agent=agent2, tools=tools2, verbose=True)

In [35]:
agent_executor2.invoke({"input": "Fotosentez nedir?"})



> Entering new AgentExecutor chain...
Fotosentez, bitkilerin, bazı bakterilerin ve alglerin güneş ışığını kullanarak karbondioksit ve suyu organik bileşenlere (genellikle glikoza) dönüştürdüğü bir kimyasal süreçtir. Bu süreçte, klorofil adı verilen yeşil pigmentler güneş ışığını absorbe eder ve bu enerji, su ve karbondioksitin glikoz ve oksijene dönüştürülmesinde kullanılır.

Fotosentez genellikle iki ana aşamadan oluşur:

1. **Işık Reaksiyonları:** Bu aşama, güneş ışığının enerjisini kullanarak suyu parçalar ve oksijen üretir. Bu süreçte ATP (adenosin trifosfat) ve NADPH (nikotinamid adenin dinükleotid fosfat) gibi enerji taşıyan moleküller üretilir.

2. **Karanlık Reaksiyonları (Calvin Döngüsü):** Bu aşamada, ışık enerjisi kullanılmadan, atmosferdeki karbondioksit, ATP ve NADPH'nin yardımıyla glikoza dönüştürülür.

Fotosentez, yeryüzündeki yaşam için son derece önemlidir çünkü bitkiler, algler ve bazı bakteriler, atmosferdeki karbondioksiti kullanarak oksijen üretir ve bu da diğer 

{'input': 'Fotosentez nedir?',
 'output': "Fotosentez, bitkilerin, bazı bakterilerin ve alglerin güneş ışığını kullanarak karbondioksit ve suyu organik bileşenlere (genellikle glikoza) dönüştürdüğü bir kimyasal süreçtir. Bu süreçte, klorofil adı verilen yeşil pigmentler güneş ışığını absorbe eder ve bu enerji, su ve karbondioksitin glikoz ve oksijene dönüştürülmesinde kullanılır.\n\nFotosentez genellikle iki ana aşamadan oluşur:\n\n1. **Işık Reaksiyonları:** Bu aşama, güneş ışığının enerjisini kullanarak suyu parçalar ve oksijen üretir. Bu süreçte ATP (adenosin trifosfat) ve NADPH (nikotinamid adenin dinükleotid fosfat) gibi enerji taşıyan moleküller üretilir.\n\n2. **Karanlık Reaksiyonları (Calvin Döngüsü):** Bu aşamada, ışık enerjisi kullanılmadan, atmosferdeki karbondioksit, ATP ve NADPH'nin yardımıyla glikoza dönüştürülür.\n\nFotosentez, yeryüzündeki yaşam için son derece önemlidir çünkü bitkiler, algler ve bazı bakteriler, atmosferdeki karbondioksiti kullanarak oksijen üretir ve b